In [9]:
import re
import pandas as pd
from random import choice

from ruwordnet import RuWordNet

from pymystem3 import Mystem

from pymorphy3 import MorphAnalyzer

from sklearn.model_selection import train_test_split

In [10]:
df = pd.read_csv('ru_cefr_short.csv')
df

,fragment,textbook-assigned cefr level
0,"Весной, летом и осенью почти каждую субботу он...",1
1,"Все говорят, что мама хорошая хозяйка. А ещё н...",1
2,На каждой двери красные плакаты и красные фона...,1
3,"Я считаю деньги, в час обедаю в кафе, а потом ...",1
4,Магазин «Чёрный квадрат» открывается в 9 часов...,1
...,...,...
7317,Утечка мозгов стала ключевым трендом междунаро...,6
7318,"По оценкам менеджеров «Промы», такая ситуация ...",6
7319,"Но это не мы, а техно-мемы заполоняют мир благ...",6
7320,Mapillary использует программное обеспечение д...,6


In [11]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['fragment'].values,
    df['textbook-assigned cefr level'].values,
    test_size=0.2,
    random_state=42,
    stratify=df['textbook-assigned cefr level']
)

len(train_texts)

5857

In [12]:
texts_to_augment = []

for i in range(len(train_labels)):
  if train_labels[i] == 6:
    texts_to_augment.append(train_texts[i])


len(texts_to_augment)

120

In [13]:
def tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)

def detokenize(tokens):
    text = ""
    for i, tok in enumerate(tokens):
        if i == 0:
            text += tok
            continue

        if tok in ",.!?:;)]}»":
            text += tok
        elif tokens[i - 1] in "([«":
            text += tok
        else:
            text += " " + tok
    return text

In [14]:
m = Mystem()

def lemmatize(text):
  lemmas = m.lemmatize(text)
    
  lemmas = [i for i in lemmas if i != ' ' and i != '\n']

  return lemmas

In [15]:
morph = MorphAnalyzer()

GOOD_POS = {
    "NOUN",   
    "ADJF", 
    "ADJS",
    "VERB",   
    "INFN"   
}

def is_replaceable_word(word):
    parses = morph.parse(word)
    if not parses:
        return False

    p = parses[0]
    pos = p.tag.POS

    if pos not in GOOD_POS:
        return False

    # исключаем местоимения и местоименные слова
    if pos == "NPRO":
        return False

    # исключаем местоименное прилагательное: этот, мой, какой, всякий
    if "Apro" in p.tag:
        return False

    return True

In [16]:
morph = MorphAnalyzer()

wn = RuWordNet("/home/tyumen/baseline_libs/myenv/lib/python3.12/site-packages/ruwordnet/static/ruwordnet-2021.db")

clean_title = lambda title: re.sub(r'\s*\([^)]*\)', '', title).strip()


def get_antonyms(word):
    parse = morph.parse(word)[0]
    lemma = parse.normal_form.lower()

    antonyms = []

    for sense in wn.get_senses(lemma):
        synset = sense.synset
        if synset is None:
            continue

        antonym_synsets = getattr(synset, "antonyms", [])

        for ant_synset in antonym_synsets:
            for ant_sense in ant_synset.senses:
                antonym = ant_sense.name.lower()
                antonym = clean_title(antonym)

                if " " in antonym:
                    continue
                if antonym == lemma:
                    continue

                antonyms.append(antonym)

    antonyms = list(dict.fromkeys(antonyms))
    return antonyms

In [17]:
with open('new_vocab_a1.txt', 'r'):
    file = open('new_vocab_a1.txt')
    a1 = file.readlines()

with open('new_vocab_a2.txt', 'r'):
    file = open('new_vocab_a2.txt')
    a2 = file.readlines()

with open('new_vocab_b1.txt', 'r'):
    file = open('new_vocab_b1.txt')
    b1 = file.readlines()

with open('new_vocab_b2.txt', 'r'):
    file = open('new_vocab_b2.txt')
    b2 = file.readlines()

with open('new_vocab_c1.txt', 'r'):
    file = open('new_vocab_c1.txt')
    c1 = file.readlines()


a1 = set([w.replace('\ufeff', '').strip() for w in a1 if w.strip()])
a2 = set([w.replace('\ufeff', '').strip() for w in a2 if w.strip()])
b1 = set([w.replace('\ufeff', '').strip() for w in b1 if w.strip()])
b2 = set([w.replace('\ufeff', '').strip() for w in b2 if w.strip()])
c1 = set([w.replace('\ufeff', '').strip() for w in c1 if w.strip()])

print(len(a1), len(a2),len(b1), len(b2), len(c1))


only_a1 = a1
only_a2 = a2 - a1
only_b1 = b1 - a2 - a1
only_b2 = b2 - b1 - a2 - a1
only_c1 = c1 - b2 - b1 - a2 - a1

print(len(only_a1), len(only_a2), len(only_b1), len(only_b2), len(only_c1))

923 1493 2821 5867 11961
923 580 1334 3142 6441


In [18]:
def rank_antonyms(antonyms):
    c1 = []
    b2 = []
    b1 = []
    a2 = []
    a1 = []
    
    for antonym in antonyms:
        if antonym in only_c1:
            c1.append(antonym)
        elif antonym in only_b2:
            b2.append(antonym)
        elif antonym in only_b1:
            b1.append(antonym)
        elif antonym in only_a2:
            a2.append(antonym)
        elif antonym in only_a1:
            a1.append(antonym)

    ranked_antonyms = c1 + b2 + b1+ a2+ a1
    return ranked_antonyms

In [19]:
INFLECTABLE_GRAMMEMES = {
    # число
    "sing", "plur",

    # род
    "masc", "femn", "neut",

    # падеж
    "nomn", "gent", "datv", "accs", "ablt", "loct",
    "gen1", "gen2", "acc2", "loc1", "loc2", "voct",

    # одуш/неодуш
    "anim", "inan",

    # глагольные категории
    "past", "pres", "futr",
    "1per", "2per", "3per",
    "indc", "impr",

    # вид
    "perf", "impf",

    # для кратких/полных прилагательных и т.п.
    "Supr", "COMP"
}

def preserve_case(source, target):
    if source.isupper():
        return target.upper()
    if source[0].isupper():
        return target.capitalize()
    return target

def inflect_antonym_like(source_word, ranked_antonyms):
    source_parses = morph.parse(source_word)
    if not source_parses:
        return None
    source_parse = source_parses[0]

    for antonym_lemma in ranked_antonyms:
        antonym_parses = morph.parse(antonym_lemma)
        if antonym_parses:
            source_pos = source_parse.tag.POS
        
            antonym_parse = None
            for p in antonym_parses:
                if p.tag.POS == source_pos:
                    antonym_parse = p
                    break
        
            if antonym_parse is None:
                antonym_parse = antonym_parses[0]
        
            # вытаскиваем у исходного слова нужные граммемы
            source_grammemes = set(source_parse.tag.grammemes)
            needed_grammemes = source_grammemes & INFLECTABLE_GRAMMEMES
        
            # пробуем поставить синоним в ту же форму
            inflected = antonym_parse.inflect(needed_grammemes)

            if inflected:
                return preserve_case(source_word, inflected.word)
        

    return None

In [20]:
augmented_texts = []
texts = []

for text in texts_to_augment:
    tokens = tokenize(text)
    words_to_replace = {}

    for idx, word in enumerate(tokens):
        if is_replaceable_word(tokens[idx]):
            words_to_replace[idx] = word

    words_to_antonyms = {}

    to_print = {}
    
    for key in words_to_replace.keys():
        antonyms = get_antonyms(tokens[key])
        ranked_antonyms = rank_antonyms(antonyms)

        if len(ranked_antonyms) > 0:

            new_form = inflect_antonym_like(words_to_replace[key], ranked_antonyms)
            if new_form:
                words_to_antonyms[key] = new_form
            
    new_text = []

    for i in range(len(tokens)):
        if i in words_to_antonyms.keys():
            new_text.append(words_to_antonyms[i])
        else:
            new_text.append(tokens[i])

    new_text = detokenize(new_text)

    if new_text != text:
        print(text)
        print(new_text)
        print('\n')

        augmented_texts.append(new_text)
        texts.append(text)

3:38–3:54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.
3: 38 – 3: 54 Собственно, сама по себе радиация незаразна. Например, мощными дозами излучения от тех же самых изотопов часто стерилизуют фрукты и овощи, а также лечат людей.


«Сразу после несчастного случая я вспомнил про видео с какими-то невероятными протезами — ловкими, красивыми, управляемыми силой мысли.»
«Сразу после несчастного случая я вспомнил про видео с какими - то невероятными протезами — неловкими, дурными, управляемыми силой мысли.»


Директор Института социального анализа и прогнозирования РАНХиГС Татьяна Малева объясняет, что в мире сегодня действительно обострилась конкуренция за человеческий капитал, это основной ресурс и мотор развития.
Директор Института социального анализа и прогнозирования РАНХиГС Татьяна Малева объясняет, что в мире сегодня действительно обострилась конкуренция за бесчеловечный 

In [21]:
len(augmented_texts)

71

In [22]:
aug_df = pd.DataFrame()
aug_df['text'] = texts
aug_df['augmented-text'] = augmented_texts
aug_df

,text,augmented-text
0,"3:38–3:54 Собственно, сама по себе радиация не...","3: 38 – 3: 54 Собственно, сама по себе радиаци..."
1,«Сразу после несчастного случая я вспомнил про...,«Сразу после несчастного случая я вспомнил про...
2,Директор Института социального анализа и прогн...,Директор Института социального анализа и прогн...
3,Почему мы ещё долго не увидим самоуправляемые ...,Почему мы ещё долго не увидим самоуправляемые ...
4,0:25–1:24 Большинство людей при этом слове впа...,0: 25 – 1: 24 Большинство людей при этом слове...
...,...,...
66,"При этом определение того, что является «разум...","При этом определение того, что является «бессм..."
67,"От одного банана — в два раза больше (0,1), а ...","От одного банана — в два раза больше (0, 1), а..."
68,00:00:24 А сегодня испытания. 13 участников. Ж...,00: 00: 24 А сегодня испытания. 13 участников....
69,«Прома» способна предложить дилерам вполне бож...,«Прома» способна предложить дилерам вполне бож...


In [23]:
aug_df.to_csv('c2_augmented_antonyms.csv')